In [1]:
import logging

import sys
from pathlib import Path

# Assume this notebook is in project_root/scripts/runner.ipynb
project_root = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parents[0]
sys.path.append(str(project_root / "src"))
sys.path.append(str(project_root / "config"))

from pipeline.synthea_runner import SyntheaRunner
from pipeline.bq_loader import BigQueryLoader
from pipeline.bq_transformer import BigQueryTransformer
from pipeline.dictionary_builder import DictionaryBuilder

logging.basicConfig(level=logging.INFO)

In [ ]:
# Cell 2: choose profile and run Synthea
config_path = str(project_root / "config" / "synthea_config.json")

# choose one of: "mock", "train", "test"
profile_name = "mock"

runner, run_params = SyntheaRunner.from_profile(
    config_path=config_path,
    profile_name=profile_name,
)

csv_dir = runner.run(**run_params)
csv_dir


INFO:pipeline.synthea_runner:Running Synthea with command: java -jar F:\Synthea\synthea-with-dependencies.jar -s 100 -cs 100 -p 100 --exporter.csv.export=true --exporter.years_of_history=2 California
INFO:pipeline.synthea_runner:Synthea working directory: F:\Synthea
INFO:pipeline.synthea_runner:Synthea CSV directory: F:\Synthea\output\csv
INFO:pipeline.synthea_runner:Profile output_dir: D:\Python Projects\Hospital readmission risk\data\raw\Synthea\mock
INFO:pipeline.synthea_runner:Files to copy: {'patients.csv', 'careplans.csv', 'encounters.csv', 'medications.csv', 'conditions.csv', 'procedures.csv', 'claims.csv', 'organizations.csv'}
INFO:pipeline.synthea_runner:Delete source files after copy: True
INFO:pipeline.synthea_runner:Copying Synthea CSVs from F:\Synthea\output\csv to D:\Python Projects\Hospital readmission risk\data\raw\Synthea\mock
INFO:pipeline.synthea_runner:Copied F:\Synthea\output\csv\patients.csv -> D:\Python Projects\Hospital readmission risk\data\raw\Synthea\mock\pat

WindowsPath('D:/Python Projects/Hospital readmission risk/data/raw/Synthea/mock')

In [ ]:
# Cell 3: Initialize BigQueryLoader and load raw_data onto BigQuery
config_path_bq = str(project_root / "config" / "bigquery_config.json")

profile_name = "mock"  # "train" or "test" as needed - keep it till the end of a pipeline

bq_loader, profile_cfg = BigQueryLoader.from_profile(
    config_path=config_path_bq,
    profile_name=profile_name,
)

bq_loader.load_profile_tables(profile_cfg)

INFO:pipeline.bq_loader:Loading CSVs from D:\Python Projects\Hospital readmission risk\data\raw\Synthea\mock into dataset healthcare-test-486920.mock_raw_data


INFO:pipeline.bq_loader:Creating dataset: healthcare-test-486920.mock_raw_data
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.mock_raw_data.careplans
INFO:pipeline.bq_loader:Loaded 187 rows into healthcare-test-486920.mock_raw_data.careplans
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.mock_raw_data.claims
INFO:pipeline.bq_loader:Loaded 2748 rows into healthcare-test-486920.mock_raw_data.claims
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.mock_raw_data.conditions
INFO:pipeline.bq_loader:Loaded 1490 rows into healthcare-test-486920.mock_raw_data.conditions
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.mock_raw_data.encounters
INFO:pipeline.bq_loader:Loaded 1650 rows into healthcare-test-486920.mock_raw_data.encounters
INFO:pipeline.bq_loader:Loading CSV into BigQuery table: healthcare-test-486920.mock_raw_data.medications
INFO:pipeline.bq_loader:Loaded

In [ ]:
# Cell 4: Initialize BigQueryTransformer and create slim tables
recipe_path = str(project_root / "config" / "bigquery_recipes.json")

transformer, t_cfg = BigQueryTransformer.from_profile(
    config_path=config_path_bq,
    profile_name=profile_name,
)
recipes_id = 0
transformer.run_query_sequence(
    recipe_path=recipe_path,
    recipes_id=recipes_id,
    project_root=str(project_root),
)
recipes_id += 1

INFO:pipeline.bq_transformer:Dataset already exists: healthcare-test-486920.data_slim
INFO:pipeline.bq_transformer:Dataset already exists: healthcare-test-486920.helper_tables
INFO:pipeline.bq_transformer:Running query:
CREATE OR REPLACE TABLE healthcare-test-486920.data_slim.mock_patients_slim
  CLUSTER BY id
AS
SELECT
  id,
  birthdate,
  deathdate,
  race,
  gender
FROM healthcare-test-486920.mock_raw_data.patients
INFO:pipeline.bq_transformer:Query finished.
INFO:pipeline.bq_transformer:Running query:
CREATE OR REPLACE TABLE healthcare-test-486920.data_slim.mock_encounters_slim
  CLUSTER BY id, patient, stop
AS (
  WITH
    end_date AS (
      SELECT MAX(stop) AS end_ts
      FROM healthcare-test-486920.mock_raw_data.encounters
    ),
    p AS (
      SELECT
        id AS patient_id,
        deathdate
      FROM healthcare-test-486920.data_slim.mock_patients_slim
    )
  SELECT
    e.id,
    e.start,
    e.stop,
    e.patient,
    e.organization,
    e.encounterclass,
    e.base_en

In [ ]:
# Cell 5: Initialize DictionaryBuilder and build local dictionaries
io_config_path = str(project_root / "config" / "dictionary_io_config.json")

builder = DictionaryBuilder(
    transformer=transformer,
    io_config_path=io_config_path,
    config_path_bq=config_path_bq,
    profile_name=profile_name,
)

builder.build_diagnoses_dictionary(profile_name)
builder.build_procedures_dictionary(profile_name)
builder.build_main_diagnoses(profile_name)

In [ ]:
# Cell 6: Load dictionaries
bq_loader.load_dictionaries("dictionaries_dir")

In [ ]:
# Cell 7: Build and load careplans
builder.build_careplans_related_diagnoses(profile_name)
bq_loader.load_dictionaries("careplans_dir")

In [ ]:
# Cell 8: Build helpers, sanity check them, build and load related diagnoses, build index stay
"""Insert data sanity check somewhere"""
transformer.run_query_sequence(
    recipe_path=recipe_path,
    recipes_id=recipes_id,
    project_root=str(project_root),
)
recipes_id += 1

checks_dir = str(project_root / "sql" / "checks")
transformer.run_helper_clinical_sanity_checks(checks_base_dir=checks_dir)
transformer.run_helper_cost_sanity_checks(checks_base_dir=checks_dir)
transformer.run_helper_utilization_sanity_checks(checks_base_dir=checks_dir)

builder.build_related_diagnoses(profile_name)
bq_loader.load_dictionaries("related_dir")

transformer.run_query_sequence(
    recipe_path=recipe_path,
    recipes_id=recipes_id,
    project_root=str(project_root),
)

In [2]:
"""Shortcut initializing everything"""
config_path_bq = str(project_root / "config" / "bigquery_config.json")
recipe_path = str(project_root / "config" / "bigquery_recipes.json")
io_config_path = str(project_root / "config" / "dictionary_io_config.json")

profile_name = "mock"  # or "train" or "test"

transformer, t_cfg = BigQueryTransformer.from_profile(
    config_path=config_path_bq,
    profile_name=profile_name,
)
builder = DictionaryBuilder(
    transformer=transformer,
    io_config_path=io_config_path,
    config_path_bq=config_path_bq,
    profile_name="mock",
)
bq_loader, profile_cfg = BigQueryLoader.from_profile(
    config_path=config_path_bq,
    profile_name=profile_name,
)


INFO:pipeline.bq_transformer:Dataset already exists: healthcare-test-486920.data_slim
INFO:pipeline.bq_transformer:Dataset already exists: healthcare-test-486920.helper_tables
